# Soup Kernel Challenge: developer check (Kaggle T4)

Fast checks of the solution before the full measured run (`templates/kaggle_notebook.ipynb`):
1. CPU model tests: every kernel body in the NumPy emulator, the dequant / weight-pipe / activation-store schedules on a model of CUDA streams and events,
2. kernel unit tests against PyTorch / bitsandbytes (`pytest solution`),
3. end-to-end comparison of the fast path with the stock transformers+PEFT step on the real model (`dev/check_fast_path.py`: loss, entropy, accuracy, rewind losses, all LoRA gradients; the load hooks bit-exact against a normal load and the load-phase GPU peak; the whole integration path `soup train` takes (hooked load, fp16 embedding, LoRA created on the host) against a normally loaded runner, and the v5-placement fallback; the start-up self-test; step time / peak memory of the v6 placement against variants; PCIe both ways; where the GPU waits; a SUMMARY block at the end),
4. one measured `soup train` run of the solution alone plus a profiled run (`harness/run_task.py --only solution --profile`).

Settings: **GPU T4 x2**, **Internet on**. Run All (~12 min). Please send back the whole log.


In [ ]:
REPO_URL = "https://github.com/shozy-frenia/decentrik_prime.git"  # your repository
COMMIT = ""  # empty = default branch


In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import base64, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret:", type(exc).__name__)
auth = []
if token:
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
r = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"], capture_output=True, text=True)
if r.returncode: raise SystemExit(r.stderr)
if COMMIT: subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
!cd /kaggle/working/repo && python dev/test_emu.py 2>&1 | tail -3
!cd /kaggle/working/repo && python dev/test_ws_schedule.py 2>&1 | tail -12
!cd /kaggle/working/repo && python dev/test_act_store.py 2>&1 | tail -3

In [ ]:
!cd /kaggle/working/repo && python task/prepare_data.py
!cd /kaggle/working/repo && python -m pytest solution -q -x 2>&1 | tail -30

In [ ]:
!cd /kaggle/working/repo && python dev/check_fast_path.py 2>&1 | grep -v Warning | tail -400

In [ ]:
!cd /kaggle/working/repo && python harness/run_task.py --only solution --profile --out runs_prof 2>&1 | tail -6
!cd /kaggle/working/repo && grep "\[solution\]" runs_prof/solution_1/train.log | head -120

In [ ]:
import json
r = json.load(open("/kaggle/working/repo/runs_prof/result.json"))
run = r["runs"]["solution"][0]
m = run["metrics"]
own = sum(m["own_kernel_ms"].values())
print({"wall_s": run["wall_s"], "gpu_peak_mib": run["gpu_peak_mib"], "train_loop_s": m["train_loop_s"],
       "median_step_s": m["median_step_s"], "apply_s": m["apply_s"], "torch_max_alloc_mib": m["torch_max_allocated_mib"],
       "torch_max_reserved_mib": m["torch_max_reserved_mib"], "own_kernel_share": round(own / 1000 / m["timed_loop_s"], 3),
       "eval_loss": r["evals"].get("solution", {}).get("mean_loss")})
print("own kernel ms:", m["own_kernel_ms"])
print("loss curve:", [round(x, 4) for x in run["log"]["loss"][:10]], "...")
prof = json.load(open("/kaggle/working/repo/runs_prof/solution_profile/profile.json"))
steps = prof["active_steps"]
busy = prof["total_gpu_us"] / steps / 1000
print(f"\nGPU kernel time per profiled step: {busy:.1f} ms (median wall step {1000 * m['median_step_s']:.1f} ms; "
      f"profiler adds CPU overhead, so compare ratios)")
for row in prof["kernels"][:50]:
    print(f"{row['time_us'] / steps / 1000:8.2f} ms/step {row['count'] / steps:7.1f}/step  {row['name'][:120]}")
